# Keypoint extraction with visualizations

Run pose inference on images in `data/test`, save normalized keypoints as CSV files, and save every annotated image under the `TRAIN` and `TEST` output folders without displaying images in the notebook.

In [11]:
from pathlib import Path

import cv2

import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import numpy as np
import pandas as pd

In [12]:
# Resolve paths from either the project root or the code directory.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

INPUT_DIR = PROJECT_ROOT / "data" / "raw"/"DATASET"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
VISUALIZATION_DIR = OUTPUT_DIR / "keypoints_visualizations"
MEDIAPIPE_DIR = PROJECT_ROOT / "model" / "mediapipe"

if not INPUT_DIR.exists():
    raise FileNotFoundError(f"Input directory does not exist: {INPUT_DIR}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
VISUALIZATION_DIR.mkdir(parents=True, exist_ok=True)

model_path = str(MEDIAPIPE_DIR / "pose_landmarker_full.task")
if not Path(model_path).is_file():
    raise FileNotFoundError(f"MediaPipe model not found: {model_path}")

base_options = python.BaseOptions(model_asset_path=model_path)
options = vision.PoseLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE,
    num_poses=1,
    output_segmentation_masks=False,
)

pose_detector = vision.PoseLandmarker.create_from_options(options)

splits = ["TRAIN", "TEST"]
image_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
CONFIDENCE_THRESHOLD = 0.25
TORSO_MULTIPLIER = 2.5

In [13]:
# MediaPipe Pose provides 33 landmarks, indexed from 0 through 32.
KPS_COUNT = 33

def extract_pose_keypoints(image_bgr):
    height, width = image_bgr.shape[:2]

    rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

    results = pose_detector.detect(mp_image)

    if not results.pose_landmarks:
        return np.full((KPS_COUNT, 3), np.nan, dtype=np.float32)

    landmarks = results.pose_landmarks[0]
    pose_keypoints = np.zeros((KPS_COUNT, 3), dtype=np.float32)

    for index, landmark in enumerate(landmarks[:KPS_COUNT]):

        pose_keypoints[index, 0] = landmark.x * width
        pose_keypoints[index, 1] = landmark.y * height
        pose_keypoints[index, 2] = landmark.z * width

    return pose_keypoints

In [14]:
def normalize_keypoints(keypoints, visibility, confidence_threshold=CONFIDENCE_THRESHOLD):
    """Normalize all 33 MediaPipe [x, y, z] landmarks."""
    normalized = keypoints.astype(np.float32, copy=True)
    xy = normalized[:, :2]
    z = normalized[:, 2]
    reliable = visibility >= confidence_threshold

    if not reliable.any():
        normalized[:, :] = 0.0
        return normalized, "none"

    hips_reliable = reliable[23] and reliable[24]
    shoulders_reliable = reliable[11] and reliable[12]
    if hips_reliable:
        center = (xy[23] + xy[24]) / 2.0
        center_z = (z[23] + z[24]) / 2.0
        center_type = "hip_center"
    elif shoulders_reliable:
        center = (xy[11] + xy[12]) / 2.0
        center_z = (z[11] + z[12]) / 2.0
        center_type = "shoulder_center"
    else:
        center = xy[reliable].mean(axis=0)
        center_z = z[reliable].mean()
        center_type = "full_body_center"

    body_radius = np.linalg.norm(xy[reliable] - center, axis=1).max()
    if hips_reliable and shoulders_reliable:
        shoulder_center = (xy[11] + xy[12]) / 2.0
        hip_center = (xy[23] + xy[24]) / 2.0
        torso_size = np.linalg.norm(shoulder_center - hip_center)
        pose_scale = max(body_radius, TORSO_MULTIPLIER * torso_size)
    else:
        pose_scale = body_radius

    if pose_scale <= np.finfo(np.float32).eps:
        normalized[:, :] = 0.0
        return normalized, "none"

    normalized[:, :2] = (xy - center) / pose_scale
    normalized[:, 2] = (z - center_z) / pose_scale
    normalized[~reliable, :] = 0.0
    return normalized, center_type


In [15]:
kp_cols = [f"kp_{i}_{axis}" for i in range(KPS_COUNT) for axis in ("x", "y", "z")]
columns = ["image_name", "width", "height"] + kp_cols + ["center_type", "label"]

for split in splits:
    rows = []
    split_visualization_dir = VISUALIZATION_DIR / split
    split_visualization_dir.mkdir(parents=True, exist_ok=True)

    for category_dir in sorted((INPUT_DIR / split).glob("*")):
        if not category_dir.is_dir():
            continue
        category = category_dir.name

        for image_path in sorted(category_dir.iterdir()):
            if image_path.suffix.lower() not in image_extensions:
                continue

            image_bgr = cv2.imread(str(image_path))
            if image_bgr is None:
                continue

            height, width = image_bgr.shape[:2]

            rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            results = pose_detector.detect(mp_image)

            if results.pose_landmarks:
                landmarks = results.pose_landmarks[0]
                keypoints = np.zeros((KPS_COUNT, 3), dtype=np.float32)

                visibility = np.zeros(KPS_COUNT, dtype=np.float32)
                for index, landmark in enumerate(landmarks[:KPS_COUNT]):
                    keypoints[index, 0] = landmark.x * width
                    keypoints[index, 1] = landmark.y * height
                    keypoints[index, 2] = landmark.z * width
                    visibility[index] = landmark.visibility

                normalized_keypoints, center_type = normalize_keypoints(keypoints, visibility)
                kp_data = normalized_keypoints.reshape(-1)
            else:
                kp_data = np.full(KPS_COUNT * 3, np.nan, dtype=np.float32)
                center_type = "none"

            image_name = (Path("data") / split / category / image_path.name).as_posix()
            rows.append([image_name, width, height, *kp_data, center_type, category])

            annotated_rgb = rgb.copy()
            if results.pose_landmarks:
                vision.drawing_utils.draw_landmarks(
                    annotated_rgb,
                    results.pose_landmarks[0],
                    vision.PoseLandmarksConnections.POSE_LANDMARKS,
                    landmark_drawing_spec=vision.drawing_styles.get_default_pose_landmarks_style(),
                    connection_drawing_spec=vision.drawing_utils.DrawingSpec(
                        color=(255, 0, 0), thickness=2
                    ),
                )

            annotated_bgr = cv2.cvtColor(annotated_rgb, cv2.COLOR_RGB2BGR)
            annotated_path = split_visualization_dir / category / image_path.name
            annotated_path.parent.mkdir(parents=True, exist_ok=True)
            if not cv2.imwrite(str(annotated_path), annotated_bgr):
                print(f"Warning: unable to save {annotated_path}")

    output_path = OUTPUT_DIR / f"keypoints_{split.lower()}.csv"
    pd.DataFrame(rows, columns=columns).to_csv(output_path, index=False)
    print(f"Saved {len(rows)} rows to {output_path}")
    print(f"Saved annotated images to {split_visualization_dir}")

pose_detector.close()

Saved 1081 rows to c:\Users\vgohu\Desktop\yoga_project\data\processed\keypoints_train.csv
Saved annotated images to c:\Users\vgohu\Desktop\yoga_project\data\processed\keypoints_visualizations\TRAIN
Saved 470 rows to c:\Users\vgohu\Desktop\yoga_project\data\processed\keypoints_test.csv
Saved annotated images to c:\Users\vgohu\Desktop\yoga_project\data\processed\keypoints_visualizations\TEST


In [ ]:
#1081 training data , 223 downdog, 180 goddess, 266 plank, 160 tree, 252 warrior2
#470 testing data , 97 downdog, 80 goddess, 115 plank, 69 tree, 109 warrior2

Outputs:

- `data/processed/keypoints_train.csv` and `data/processed/keypoints_test.csv`
- Annotated images under `data/processed/keypoints_visualizations/<split>/<label>/`